# LDA topic model for ECOSOC transcripts

**Owner:** Tahvia. **Due:** May 10. **Dependents:** Vadim swaps `topic_assignments.parquet` into `02_alignment_descriptive.ipynb`.

Trains LDA at k = 6 / 10 / 15 on the year-level TF-IDF matrix, picks best by `u_mass` coherence, labels topics by cosine similarity to hand-built lexicons, clusters years into 4 KMeans archetypes, and saves five figures for §4.1 of the paper.

Constants (boilerplate filter, lexicons, topic-to-theme mapping, colors) live in `topic_helpers.py` so this notebook and `label_transcripts.ipynb` stay in sync.

**Inputs:** `data/features/tier2_tfidf.parquet`  
**Outputs:** `data/interim/topic_assignments.parquet`, `data/interim/lda_model/lda.model`, five figures in `output/figures/`

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from gensim import corpora, models
from gensim.models.coherencemodel import CoherenceModel
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

from topic_helpers import (
    DATA, FIG_DIR,
    filter_vocab,
    AGENCY_LEXICONS, SECTOR_LEXICONS,
    TOPIC_TO_THEME, THEME_COLORS,
    lexicon_vector, topic_vector,
)

FIG_DIR.mkdir(parents=True, exist_ok=True)
(DATA / 'interim' / 'lda_model').mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 150})

## Step 1. Load TF-IDF and drop boilerplate columns

The raw tier2 matrix contains about 22 n-grams from the ECOSOC meeting-record header template. They are filtered out here so LDA learns substantive content.

In [2]:
df = pd.read_parquet(DATA / 'features' / 'tier2_tfidf.parquet')
all_cols = [c for c in df.columns if c.startswith('t_')]
vocab, dropped = filter_vocab(all_cols)
X = df[vocab].values

print(f'df shape:        {df.shape}')
print(f'dropped tokens:  {len(dropped)} -> {dropped[:5]}...')
print(f'vocab after:     {len(vocab)}')
print(f'X shape:         {X.shape}')

df shape:        (26, 2002)
dropped tokens:  22 -> ['t_750 united', 't_chief official', 't_contents agenda', 't_corrections record', 't_council provisional']...
vocab after:     1978
X shape:         (26, 1978)


## Step 2. Train LDA at k = 6, 10, 15 and pick the best

Using `u_mass` coherence because the input is year-level TF-IDF rather than tokenized per-meeting texts. Scores are negative; less-negative is better, so `max` still picks the best k.

In [3]:
corpus = [[(i, float(v)) for i, v in enumerate(row) if v > 0] for row in X]
dictionary = corpora.Dictionary([[w] for w in vocab])

scores = {}
for k in [6, 10, 15]:
    m = models.LdaMulticore(corpus, num_topics=k, id2word=dictionary,
                            passes=20, workers=2, random_state=42)
    coh = CoherenceModel(model=m, corpus=corpus, dictionary=dictionary,
                         coherence='u_mass').get_coherence()
    scores[k] = (m, coh)
    print(f'k={k:2d}   coherence={coh:.4f}')

best_k = max(scores, key=lambda k: scores[k][1])
lda = scores[best_k][0]
print(f'\nBest k: {best_k}')

k= 6   coherence=-0.1149
k=10   coherence=-0.1267
k=15   coherence=-0.2392

Best k: 6


## Step 3. Inspect top words per topic

Read these and judge whether topics are meaningful or boilerplate.

In [4]:
for i in range(best_k):
    words = [w for w, _ in lda.show_topic(i, topn=20)]
    print(f'Topic {i} ({TOPIC_TO_THEME[i]}): {words}')

Topic 0 (development): ['t_resident coordinator', 't_basic', 't_aid', 't_promoting', 't_oda', 't_capacities', 't_markets', 't_humanitarian assistance', 't_south south', 't_national development', 't_development partners', 't_coordinator', 't_communities', 't_developed countries', 't_teams', 't_draft resolution', 't_resident', 't_structural', 't_2030 agenda', 't_analysis']
Topic 1 (development): ['t_developed countries', 't_aid', 't_climate change', 't_summit', 't_promoting', 't_world bank', 't_draft resolution', 't_emergency', 't_millennium', 't_economic growth', 't_draft decision', 't_donors', 't_humanitarian assistance', 't_2030 agenda', 't_crises', 't_resident', 't_question', 't_communities', 't_coordinator', 't_th']
Topic 2 (climate): ['t_summit', 't_turkey', 't_agenda item', 't_scale', 't_communities', 't_coordinator', 't_employment', 't_emergency', 't_school', 't_climate change', 't_developed countries', 't_donors', 't_2030 agenda', 't_draft resolution', 't_ministerial', 't_th', '

### Validation flag (kept for the paper's §5)

The current run shows four of six topics sharing heavy vocabulary (`developed countries`, `aid`, `climate change`, `2030 agenda`). LDA at k = 6 is splitting one development-discourse blob into near-duplicates. Only Topic 5 (UNAIDS, epidemic, pandemic) is cleanly distinct. KMeans on the mixtures produces a near-degenerate split. These limitations are acknowledged in §5 of the paper.

## Step 4. Map topics to agencies and sectors via cosine similarity

In [5]:
topic_to_agency, topic_to_sector = {}, {}
for i in range(best_k):
    tv = topic_vector(lda, i, vocab).reshape(1, -1)
    agency_sims = {a: cosine_similarity(tv, lexicon_vector(toks, vocab).reshape(1, -1))[0][0]
                   for a, toks in AGENCY_LEXICONS.items()}
    sector_sims = {s: cosine_similarity(tv, lexicon_vector(toks, vocab).reshape(1, -1))[0][0]
                   for s, toks in SECTOR_LEXICONS.items()}
    topic_to_agency[i] = max(agency_sims, key=agency_sims.get)
    topic_to_sector[i] = max(sector_sims, key=sector_sims.get)
    print(f'Topic {i} -> agency: {topic_to_agency[i]:<10} sector: {topic_to_sector[i]}')

Topic 0 -> agency: WHO        sector: health
Topic 1 -> agency: WFP        sector: environment
Topic 2 -> agency: UNFCCC     sector: environment
Topic 3 -> agency: ILO        sector: environment
Topic 4 -> agency: UNFCCC     sector: agriculture
Topic 5 -> agency: UNAIDS     sector: health


## Step 5. Per-year topic mixtures and KMeans archetypes

In [6]:
mixtures = np.array([
    [p for _, p in sorted(lda.get_document_topics(row, minimum_probability=0.0))]
    for row in corpus
])
archetypes = KMeans(n_clusters=4, random_state=42, n_init=10).fit_predict(mixtures)

print(f'mixtures shape:    {mixtures.shape}')
print(f'archetype counts:  {pd.Series(archetypes).value_counts().to_dict()}')

mixtures shape:    (26, 6)
archetype counts:  {0: 22, 3: 2, 2: 1, 1: 1}


## Step 6. Save assignments and the LDA model

In [7]:
assignments = pd.DataFrame({
    'year':          df['year'].values,
    'segment':       df['segment'].values,
    'top_topic':     [topic_to_agency[int(np.argmax(m))] for m in mixtures],
    'topic_mixture': [list(m) for m in mixtures],
    'archetype':     archetypes,
})
assignments.to_parquet(DATA / 'interim' / 'topic_assignments.parquet', index=False)
lda.save(str(DATA / 'interim' / 'lda_model' / 'lda.model'))

print(f'Saved assignments: {assignments.shape}')
assignments.head()

Saved assignments: (26, 5)


,year,segment,top_topic,topic_mixture,archetype
0,2000,SR,WFP,"[0.011571615, 0.94214195, 0.011571615, 0.01157...",2
1,2001,SR,WFP,"[0.0042199856, 0.97890013, 0.0042199865, 0.004...",0
2,2002,SR,WFP,"[0.004075938, 0.9796203, 0.004075938, 0.004075...",0
3,2003,SR,WFP,"[0.004118596, 0.979407, 0.0041185943, 0.004118...",0
4,2004,SR,WFP,"[0.0040190686, 0.9799047, 0.0040190695, 0.0040...",0


## Step 7. Figures for §4.1

Five figures total. Figures 4 and 5 require `transcripts_human_labels.csv` from `label_transcripts.ipynb`; they are skipped with a warning if missing.

In [8]:
years = assignments['year'].values
K = lda.num_topics

# ---- Figure 1: top 10 words per topic ----
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, ax in enumerate(axes.flat):
    if i >= K:
        ax.axis('off')
        continue
    ww = lda.show_topic(i, topn=10)
    words   = [w.replace('t_', '') for w, _ in ww][::-1]
    weights = [wt for _, wt in ww][::-1]
    ax.barh(range(len(words)), weights, color=THEME_COLORS[TOPIC_TO_THEME[i]])
    ax.set_yticks(range(len(words)))
    ax.set_yticklabels(words, fontsize=9)
    ax.set_title(f'Topic {i} -> {TOPIC_TO_THEME[i]}', fontsize=11, fontweight='bold')
    ax.set_xlabel('weight', fontsize=8)
fig.suptitle('Top 10 words per LDA topic (color = assigned theme)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'topic_words.png', bbox_inches='tight')
plt.close()

# ---- Figure 2: stacked-area topic mixture over time ----
order = np.argsort(years)
yrs, mix = years[order], mixtures[order]
fig, ax = plt.subplots(figsize=(12, 5))
ax.stackplot(yrs, mix.T,
             labels=[f'Topic {i} ({TOPIC_TO_THEME[i]})' for i in range(K)],
             colors=[THEME_COLORS[TOPIC_TO_THEME[i]] for i in range(K)],
             alpha=0.85)
ax.set(xlabel='Year', ylabel='Topic share',
       title='ECOSOC topic mixtures over time (2000-2025)',
       xlim=(yrs.min(), yrs.max()), ylim=(0, 1))
ax.legend(loc='center left', bbox_to_anchor=(1.01, 0.5), fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / 'topic_mixture_over_time.png', bbox_inches='tight')
plt.close()

# ---- Figure 3: KMeans archetype timeline ----
arch_colors = plt.cm.tab10(np.linspace(0, 1, 4))
fig, ax = plt.subplots(figsize=(12, 2.5))
for y, a in zip(years, assignments['archetype']):
    ax.bar(y, 1, color=arch_colors[a], width=0.9)
    ax.text(y, 0.5, str(a), ha='center', va='center',
            fontsize=8, color='white', fontweight='bold')
ax.set(yticks=[], xlabel='Year',
       title='KMeans agenda archetype by year (4 clusters)')
ax.legend(handles=[Patch(facecolor=arch_colors[i], label=f'Archetype {i}') for i in range(4)],
          loc='center left', bbox_to_anchor=(1.01, 0.5), fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / 'archetype_timeline.png', bbox_inches='tight')
plt.close()

print('Saved: topic_words.png, topic_mixture_over_time.png, archetype_timeline.png')

Saved: topic_words.png, topic_mixture_over_time.png, archetype_timeline.png


In [9]:
# ---- Figures 4 and 5: validation gate (need human labels) ----
labels_path = DATA / 'interim' / 'transcripts_human_labels.csv'
if not labels_path.exists():
    print(f'Skipping figures 4 and 5: {labels_path} not found.')
    print('Run label_transcripts.ipynb first.')
else:
    labels = pd.read_csv(labels_path)
    all_themes = sorted(THEME_COLORS.keys())

    # Figure 4: coverage comparison
    human_counts = labels['human_label'].value_counts()
    lda_top      = pd.Series([TOPIC_TO_THEME[int(np.argmax(m))] for m in mixtures]).value_counts()
    x, w = np.arange(len(all_themes)), 0.4
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.bar(x - w/2, [human_counts.get(t, 0) for t in all_themes], w,
           label=f'Human labels (n={len(labels)})', color='#3b82f6')
    ax.bar(x + w/2, [lda_top.get(t, 0) for t in all_themes], w,
           label=f'LDA top theme per year (n={len(mixtures)})', color='#f97316')
    ax.set_xticks(x)
    ax.set_xticklabels(all_themes, rotation=30, ha='right')
    ax.set_ylabel('Count')
    ax.set_title('Human labels vs LDA top theme: coverage comparison')
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'human_vs_lda_coverage.png', bbox_inches='tight')
    plt.close()

    # Figure 5: agreement by label
    year_top3 = {}
    for _, row in assignments.iterrows():
        m = row['topic_mixture']
        top3 = sorted(range(len(m)), key=lambda i: -m[i])[:3]
        year_top3[row['year']] = {TOPIC_TO_THEME[i] for i in top3}

    per_theme = {}
    for _, row in labels.iterrows():
        th = row['human_label']
        per_theme.setdefault(th, [0, 0])
        per_theme[th][1] += 1
        if th in year_top3.get(row['year'], set()):
            per_theme[th][0] += 1

    themes_sorted = sorted(per_theme, key=lambda t: -per_theme[t][1])
    rates  = [per_theme[t][0] / per_theme[t][1] for t in themes_sorted]
    totals = [per_theme[t][1] for t in themes_sorted]

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(themes_sorted, rates,
                  color=[THEME_COLORS[t] for t in themes_sorted])
    for bar, total, rate in zip(bars, totals, rates):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{rate:.0%}\n(n={total})', ha='center', va='bottom', fontsize=9)
    ax.axhline(0.6, ls='--', color='red', label='60% target')
    ax.set(ylim=(0, 1.15),
           ylabel='Match rate (human label in LDA top-3)')
    overall = sum(v[0] for v in per_theme.values()) / sum(v[1] for v in per_theme.values())
    ax.set_title(f'Validation gate by human label: overall {overall:.1%}')
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'agreement_by_label.png', bbox_inches='tight')
    plt.close()
    print('Saved: human_vs_lda_coverage.png, agreement_by_label.png')

print(f'\nAll figures in {FIG_DIR}/')

Saved: human_vs_lda_coverage.png, agreement_by_label.png

All figures in /Users/latahviawilliams/BUSN-20800-Final-Project-/output/figures/
